## 1. System Under Test — `PaymentHandler`

`PaymentHandler` — серце сервісу платежів. Приймає `OrderPlacedEvent`, перевіряє ідемпотентність, викликає платіжний шлюз, публікує `PaymentProcessed` / `PaymentFailed`.

![SUT — PaymentHandler та залежності](img/diag_01_flowchart.png)

Жовтим — те, що ми **мокаємо у юніт-тестах** (`Частина 1`). Зеленим — реальний RabbitMQ, що піднімається через TestContainers у `Частині 2` та бонусі.

In [ ]:
from __future__ import annotations

import asyncio
import json
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from decimal import Decimal
from typing import Protocol
from unittest.mock import AsyncMock

# --- Domain events ---

@dataclass(frozen=True)
class OrderPlacedEvent:
    order_id: uuid.UUID
    customer_id: str
    total_amount: Decimal
    event_id: uuid.UUID = field(default_factory=uuid.uuid4)

@dataclass(frozen=True)
class PaymentProcessedEvent:
    order_id: uuid.UUID
    transaction_id: str
    amount: Decimal

@dataclass(frozen=True)
class PaymentFailedEvent:
    order_id: uuid.UUID
    reason: str

@dataclass(frozen=True)
class PaymentRecord:
    order_id: uuid.UUID
    transaction_id: str
    amount: Decimal
    paid_at: datetime

@dataclass(frozen=True)
class ChargeResult:
    success: bool
    transaction_id: str | None = None
    error_message: str | None = None

# --- Dependencies (Protocol = duck-typed interface) ---

class PaymentRepository(Protocol):
    async def payment_exists_for_order(self, order_id: uuid.UUID) -> bool: ...
    async def save_payment(self, record: PaymentRecord) -> None: ...

class PaymentGateway(Protocol):
    async def charge(self, customer_id: str, amount: Decimal) -> ChargeResult: ...

class EventPublisher(Protocol):
    async def publish(self, event: object) -> None: ...

# --- The handler we test ---

class PaymentHandler:
    """Ідемпотентність → charge → save+publish або publish(failed)."""

    def __init__(
        self,
        repo: PaymentRepository,
        gateway: PaymentGateway,
        publisher: EventPublisher,
    ) -> None:
        self._repo = repo
        self._gateway = gateway
        self._publisher = publisher

    async def handle(self, event: OrderPlacedEvent) -> None:
        # 1. Idempotency
        if await self._repo.payment_exists_for_order(event.order_id):
            return

        # 2. External gateway call (може кинути виняток — пропагуємо вгору)
        result = await self._gateway.charge(event.customer_id, event.total_amount)

        # 3. Failure path — publish PaymentFailed, save НЕ робимо
        if not result.success:
            await self._publisher.publish(
                PaymentFailedEvent(
                    order_id=event.order_id,
                    reason=result.error_message or "unknown",
                )
            )
            return

        # 4. Success path — save + publish PaymentProcessed
        await self._repo.save_payment(
            PaymentRecord(
                order_id=event.order_id,
                transaction_id=result.transaction_id,
                amount=event.total_amount,
                paid_at=datetime.now(timezone.utc),
            )
        )
        await self._publisher.publish(
            PaymentProcessedEvent(
                order_id=event.order_id,
                transaction_id=result.transaction_id,
                amount=event.total_amount,
            )
        )

print("PaymentHandler та залежності визначено.")

## 2. Частина 1: Unit-тести (AsyncMock + AAA)

**Стратегія ізоляції:** мокаємо всі три залежності через `AsyncMock`. Перевіряємо side-effects:

| C# / NSubstitute | Python / AsyncMock |
|---|---|
| `mock.Received(1).Method(...)` | `mock.method.assert_awaited_once_with(...)` |
| `mock.DidNotReceive().Method(...)` | `mock.method.assert_not_awaited()` |
| `mock.Method(...).Returns(x)` | `mock.method.return_value = x` |
| `mock.Method(...).Throws(exc)` | `mock.method.side_effect = exc` |

![Гілки рішення PaymentHandler](img/diag_02_statediagram-v2.png)

Кожен тест слідує **AAA**-патерну (`arrange / act / assert`) — особливо важливо для EDA, бо «у нас немає такого повноцінного сценарію... ми реагуємо на сайд-ефекти, тому обов'язково в тесті пишемо, що ми хочемо зробити».

In [ ]:
def make_handler():
    """Створює PaymentHandler з трьома AsyncMock-залежностями (≡ NSubstitute ctor у lecture11.cs)."""
    repo = AsyncMock(spec=PaymentRepository)
    gateway = AsyncMock(spec=PaymentGateway)
    publisher = AsyncMock(spec=EventPublisher)
    handler = PaymentHandler(repo, gateway, publisher)
    return handler, repo, gateway, publisher

def make_event(amount: Decimal = Decimal("99.99")) -> OrderPlacedEvent:
    return OrderPlacedEvent(
        order_id=uuid.uuid4(),
        customer_id="cust-1",
        total_amount=amount,
    )

print("Фабрики тестових об'єктів готові.")

### 2.1 Тест 1 — Успішна оплата → `save_payment` + `publish(PaymentProcessed)`

- **Arrange:** `payment_exists=False`, `charge` повертає `success=True`.
- **Act:** `await handler.handle(event)`.
- **Assert:** `repo.save_payment` викликано 1× із коректним `PaymentRecord`; `publisher.publish` викликано з `PaymentProcessedEvent`.

In [ ]:
async def test_successful_payment_saves_and_publishes_processed():
    # Arrange
    handler, repo, gateway, publisher = make_handler()
    event = make_event()
    repo.payment_exists_for_order.return_value = False
    gateway.charge.return_value = ChargeResult(success=True, transaction_id="tx-1")

    # Act
    await handler.handle(event)

    # Assert — save виконано
    repo.save_payment.assert_awaited_once()
    saved: PaymentRecord = repo.save_payment.await_args.args[0]
    assert saved.order_id == event.order_id
    assert saved.transaction_id == "tx-1"
    assert saved.amount == event.total_amount

    # Assert — publish(PaymentProcessed) виконано
    publisher.publish.assert_awaited_once()
    published = publisher.publish.await_args.args[0]
    assert isinstance(published, PaymentProcessedEvent)
    assert published.order_id == event.order_id
    assert published.transaction_id == "tx-1"

await test_successful_payment_saves_and_publishes_processed()
print("✓ Тест 1 — успішна оплата: PASS")

### 2.2 Тест 2 — Невдала оплата → `publish(PaymentFailed)`, **НЕ** `save_payment`

Подвійне ствердження: `PaymentFailed` опубліковано **і** `save_payment` **не** викликано. Друге твердження критичне — інакше у БД з'явиться запис про «успішну» оплату, яка насправді відхилена шлюзом.

In [ ]:
async def test_failed_payment_publishes_failed_without_save():
    # Arrange
    handler, repo, gateway, publisher = make_handler()
    event = make_event(amount=Decimal("50"))
    repo.payment_exists_for_order.return_value = False
    gateway.charge.return_value = ChargeResult(success=False, error_message="Declined")

    # Act
    await handler.handle(event)

    # Assert — published PaymentFailed
    publisher.publish.assert_awaited_once()
    published = publisher.publish.await_args.args[0]
    assert isinstance(published, PaymentFailedEvent)
    assert published.order_id == event.order_id
    assert published.reason == "Declined"

    # Assert — НЕ збережено (ключове)
    repo.save_payment.assert_not_awaited()

await test_failed_payment_publishes_failed_without_save()
print("✓ Тест 2 — невдала оплата: PASS")

### 2.3 Тест 3 — Дублікат → **НЕ** `charge`, **НЕ** `publish`

Ідемпотентність на рівні хендлера: якщо `payment_exists_for_order` повертає `True`, виходимо негайно. Шлюз не задіяний, нічого не публікується.



In [ ]:
async def test_duplicate_skips_gateway_and_publishing():
    # Arrange — payment вже існує
    handler, repo, gateway, publisher = make_handler()
    event = make_event()
    repo.payment_exists_for_order.return_value = True

    # Act
    await handler.handle(event)

    # Assert — нічого не виконано: ані charge, ані publish, ані save
    gateway.charge.assert_not_awaited()
    publisher.publish.assert_not_awaited()
    repo.save_payment.assert_not_awaited()

await test_duplicate_skips_gateway_and_publishing()
print("✓ Тест 3 — дублікат: PASS")

### 2.4 Тест 4 — Помилка шлюзу → exception вгору, **НЕ** `publish`

Якщо `gateway.charge` піднімає виняток (`TimeoutError`, мережева помилка), хендлер **не маскує помилку** — exception пропагується наверх для retry / DLQ-handling. Нічого не публікується і не зберігається — це гарантує, що нічого не «протече» при тимчасовому збої шлюзу.

In [ ]:
async def test_gateway_error_raises_and_does_not_publish():
    # Arrange
    handler, repo, gateway, publisher = make_handler()
    event = make_event()
    repo.payment_exists_for_order.return_value = False
    gateway.charge.side_effect = TimeoutError("Gateway timeout")

    # Act & Assert — exception пропагується
    raised: TimeoutError | None = None
    try:
        await handler.handle(event)
    except TimeoutError as exc:
        raised = exc
    assert raised is not None, "Очікувано TimeoutError, але exception не піднято"
    assert "Gateway timeout" in str(raised)

    # Нічого не опубліковано і не збережено
    publisher.publish.assert_not_awaited()
    repo.save_payment.assert_not_awaited()

await test_gateway_error_raises_and_does_not_publish()
print("✓ Тест 4 — exception зі шлюзу: PASS")

### 2.5 Підсумок Частини 1

| # | Сценарій | `payment_exists` | `charge` | `save_payment` | `publish(Processed)` | `publish(Failed)` | Exception |
|---|---|:-:|:-:|:-:|:-:|:-:|:-:|
| 1 | Успіх | False | success=True | **1×** | **1×** | — | — |
| 2 | Невдача | False | success=False | — | — | **1×** | — |
| 3 | Дублікат | **True** | — | — | — | — | — |
| 4 | Шлюз падає | False | raises | — | — | — | **TimeoutError** |

Усе, що ми перевіряємо — **side-effects через моки**, бо хендлер не повертає значення (fire-and-forget). Це принципова відмінність від тестів моноліту, де часто перевіряємо `return value`.

## 3. Частина 2: Інтеграційний тест із TestContainers + RabbitMQ

**Чому реальний брокер?** Юніт-тести з моками **не побачать**:
- помилок serialization (`Decimal` → JSON → знову `Decimal`, datetime з timezone);
- неправильного routing key або binding (exchange→queue);
- проблем з ACK / re-delivery;
- неузгоджених локалей (`,` vs `.` як decimal-separator).

![Інтеграційний flow: pytest → TestContainers → RabbitMQ → consumer](img/diag_03_sequencediagram.png)

**Pitfalls:** потрібен Docker; перший старт контейнера ~10–20с — один контейнер на весь модуль через module-scope fixture.

In [ ]:
# Залежності для Частини 2 + бонусу. Якщо вже встановлено — no-op.
import importlib.util as _u
_missing = [p for p in ("aio_pika", "testcontainers") if _u.find_spec(p) is None]
if _missing:
    print(f"Встановлюю: {_missing}")
    %pip install --quiet "aio-pika>=9.4" "testcontainers[rabbitmq]>=4.0"
else:
    print("Залежності вже встановлено.")

### 3.1 Як запустити у хмарі без локального Docker

Якщо ви відкрили цей notebook у Google Colab / Kaggle / Binder — там **немає Docker daemon**, і `RabbitMqContainer.start()` впаде. Робочий шлях — Testcontainers Cloud:

| Варіант | Як |
|---|---|
| **Testcontainers Cloud** (рекомендовано для Colab) | Зареєструйтесь на [app.testcontainers.cloud](https://app.testcontainers.cloud), створіть service account token, додайте в **Colab Secrets** як `TC_CLOUD_TOKEN`, дозвольте доступ цьому notebook. Решта коду не змінюється — `testcontainers>=4.0` детектує токен автоматично і проксує контейнери у remote Docker. Free tier ~100 хв/міс. |

Наступний cell детектує середовище й підказує, якщо Docker недоступний.

In [ ]:
import os
from pathlib import Path

# Colab Secrets: якщо ви додали TC_CLOUD_TOKEN — підтягуємо у env.
try:
    from google.colab import userdata  # type: ignore
    _tok = userdata.get("TC_CLOUD_TOKEN")
    if _tok and not os.environ.get("TC_CLOUD_TOKEN"):
        os.environ["TC_CLOUD_TOKEN"] = _tok
except Exception:
    pass  # не Colab — це нормально

if os.environ.get("TC_CLOUD_TOKEN"):
    print("✓ Testcontainers Cloud token знайдено — контейнери підуть у remote Docker.")
elif Path("/var/run/docker.sock").exists() or os.environ.get("DOCKER_HOST"):
    print("✓ Локальний Docker daemon знайдено — контейнери підуть локально.")
else:
    print("⚠ Docker не знайдено. Подальші cells впадуть. Варіанти:")
    print("   1. Testcontainers Cloud: додайте TC_CLOUD_TOKEN у Colab Secrets і перезапустіть цей cell.")
    print("   2. Codespaces / локальний Docker.")
    print("   3. Colab apt fallback (див. markdown вище).")

In [ ]:
from testcontainers.rabbitmq import RabbitMqContainer
import aio_pika

# Module-scoped: один контейнер на всі тести нижче (~10-20с старт).
_rabbit = RabbitMqContainer("rabbitmq:3.13-management")
_rabbit.start()
AMQP_URL = (
    f"amqp://guest:guest@{_rabbit.get_container_host_ip()}:"
    f"{_rabbit.get_exposed_port(5672)}/"
)
print(f"RabbitMQ запущено: {AMQP_URL}")

In [ ]:
async def declare_topology(
    channel: aio_pika.abc.AbstractChannel, queue_name: str
):
    """Створює exchange 'orders' (topic) + чергу + binding на routing_key='order.placed'."""
    exchange = await channel.declare_exchange(
        "orders", aio_pika.ExchangeType.TOPIC, durable=False
    )
    queue = await channel.declare_queue(queue_name, durable=False, auto_delete=False)
    await queue.bind(exchange, routing_key="order.placed")
    return exchange, queue

def serialize(event: OrderPlacedEvent) -> bytes:
    """Domain → JSON bytes. Decimal → str (інакше json.dumps впаде)."""
    return json.dumps({
        "event_id": str(event.event_id),
        "order_id": str(event.order_id),
        "customer_id": event.customer_id,
        "total_amount": str(event.total_amount),
    }).encode()

def deserialize(body: bytes) -> OrderPlacedEvent:
    """JSON bytes → Domain. str → Decimal (зберігає точність)."""
    d = json.loads(body)
    return OrderPlacedEvent(
        event_id=uuid.UUID(d["event_id"]),
        order_id=uuid.UUID(d["order_id"]),
        customer_id=d["customer_id"],
        total_amount=Decimal(d["total_amount"]),
    )

print("Helpers готові: declare_topology, serialize, deserialize.")

### 3.2 Тест 1 — Publish OrderPlaced → consumer десеріалізує коректно

1. Створюємо exchange + queue + binding.
2. Запускаємо consumer, який кладе десеріалізовані події у `received`.
3. Publish OrderPlaced з `Decimal("199.99")` (перевірка decimal-localization).
4. Polling 50 × 100мс до появи події у `received`.
5. Assert: усі поля збігаються, `total_amount` залишився `Decimal`.

In [ ]:
async def integration_test_publish_and_deserialize():
    queue_name = f"test.deserialize.{uuid.uuid4().hex[:8]}"
    received: list[OrderPlacedEvent] = []

    connection = await aio_pika.connect_robust(AMQP_URL)
    async with connection:
        channel = await connection.channel()
        exchange, queue = await declare_topology(channel, queue_name)

        async def on_message(message: aio_pika.abc.AbstractIncomingMessage):
            async with message.process():
                received.append(deserialize(message.body))

        await queue.consume(on_message)

        # Publish
        event = OrderPlacedEvent(
            order_id=uuid.uuid4(),
            customer_id="cust-42",
            total_amount=Decimal("199.99"),
        )
        await exchange.publish(
            aio_pika.Message(serialize(event)),
            routing_key="order.placed",
        )

        # Polling — до 5с
        for _ in range(50):
            if received:
                break
            await asyncio.sleep(0.1)

        assert received, "Повідомлення не отримано за 5с"
        got = received[0]
        assert got.event_id == event.event_id
        assert got.order_id == event.order_id
        assert got.customer_id == "cust-42"
        assert got.total_amount == Decimal("199.99")
        assert isinstance(got.total_amount, Decimal), "Decimal загубився при серіалізації"

await integration_test_publish_and_deserialize()
print("✓ Інтегр. тест 1 — publish + deserialize: PASS")

### 3.3 Тест 2 — Повторна публікація → без дублікатів обробки

Сценарій: публікуємо ту саму подію (однаковий `event_id`) **двічі**. Consumer тримає in-memory `set` оброблених `event_id` — друга delivery відкидається на рівні infrastructure.

Перевіряємо: `process_order` (`AsyncMock`) викликано **рівно один раз**.

> RabbitMQ не гарантує exactly-once delivery «з коробки» — це робить consumer-логіка через `event_id` як ідемпотентний ключ. У реальній системі — або БД-`UNIQUE`, або Redis-`SET NX`.

In [ ]:
async def integration_test_republish_no_duplicates():
    queue_name = f"test.idempotency.{uuid.uuid4().hex[:8]}"
    process_order = AsyncMock()
    processed_ids: set[uuid.UUID] = set()

    connection = await aio_pika.connect_robust(AMQP_URL)
    async with connection:
        channel = await connection.channel()
        exchange, queue = await declare_topology(channel, queue_name)

        async def on_message(message: aio_pika.abc.AbstractIncomingMessage):
            async with message.process():
                event = deserialize(message.body)
                if event.event_id in processed_ids:
                    return  # idempotency на consumer-рівні
                processed_ids.add(event.event_id)
                await process_order(event)

        await queue.consume(on_message)

        # Publish ту саму подію двічі
        event = OrderPlacedEvent(
            order_id=uuid.uuid4(),
            customer_id="cust-7",
            total_amount=Decimal("75.50"),
        )
        body = serialize(event)
        await exchange.publish(aio_pika.Message(body), routing_key="order.placed")
        await exchange.publish(aio_pika.Message(body), routing_key="order.placed")

        # Чекаємо, поки перша обробиться, + grace period для другої
        for _ in range(50):
            if process_order.await_count >= 1:
                break
            await asyncio.sleep(0.1)
        await asyncio.sleep(0.5)  # дати шанс другій delivery

        assert process_order.await_count == 1, (
            f"Очікувано 1 виклик, отримано {process_order.await_count}"
        )

await integration_test_republish_no_duplicates()
print("✓ Інтегр. тест 2 — re-publish без дублікатів: PASS")

## 4. Бонус: Chaos-тест — kill consumer до ACK → redelivery

**AMQP-семантика, на яку спираємось:**
- `auto_ack=False` (manual ACK) — RabbitMQ тримає повідомлення «in-flight», поки consumer не зробить `message.ack()`.
- Якщо channel/connection закривається **без ACK** — повідомлення повертається до черги з `redelivered=True`.

![Redelivery після crash до ACK](img/diag_04_statediagram-v2.png)

Лектор пропонує робити `docker kill` контейнера consumer-а; ми робимо еквівалент через `await connection.close()` — це теж змушує RabbitMQ повернути unacked-повідомлення.

In [ ]:
async def chaos_test_consumer_crash_triggers_redelivery():
    queue_name = f"test.chaos.{uuid.uuid4().hex[:8]}"

    # --- 1. Декларуємо topology і пушимо повідомлення ---
    publisher_conn = await aio_pika.connect_robust(AMQP_URL)
    async with publisher_conn:
        channel = await publisher_conn.channel()
        exchange, _ = await declare_topology(channel, queue_name)
        event = OrderPlacedEvent(
            order_id=uuid.uuid4(),
            customer_id="cust-chaos",
            total_amount=Decimal("42.00"),
        )
        await exchange.publish(
            aio_pika.Message(serialize(event)), routing_key="order.placed"
        )

    # --- 2. Consumer A: отримує, але «падає» до ACK ---
    consumer_a_conn = await aio_pika.connect(AMQP_URL)  # non-robust — не реконектиться
    channel_a = await consumer_a_conn.channel()
    await channel_a.set_qos(prefetch_count=1)
    queue_a = await channel_a.declare_queue(queue_name, durable=False, auto_delete=False)

    a_received = asyncio.Event()

    async def consumer_a_callback(message: aio_pika.abc.AbstractIncomingMessage):
        a_received.set()
        await asyncio.sleep(10)  # «думаємо» 10с — далеко за crash
        await message.ack()       # цей ACK не встигне виконатись

    await queue_a.consume(consumer_a_callback, no_ack=False)
    await asyncio.wait_for(a_received.wait(), timeout=5.0)

    # Емулюємо crash: закриваємо connection до того, як callback зробить ACK
    await consumer_a_conn.close()
    print("  Consumer A: emulated crash before ACK")

    # --- 3. Consumer B забирає re-delivered повідомлення ---
    consumer_b_conn = await aio_pika.connect_robust(AMQP_URL)
    redelivered_flag = {"value": None}
    b_received = asyncio.Event()
    async with consumer_b_conn:
        channel_b = await consumer_b_conn.channel()
        queue_b = await channel_b.declare_queue(
            queue_name, durable=False, auto_delete=False
        )

        async def consumer_b_callback(message: aio_pika.abc.AbstractIncomingMessage):
            redelivered_flag["value"] = message.redelivered
            await message.ack()
            b_received.set()

        await queue_b.consume(consumer_b_callback, no_ack=False)
        await asyncio.wait_for(b_received.wait(), timeout=10.0)

    assert redelivered_flag["value"] is True, (
        f"redelivered має бути True, отримано {redelivered_flag['value']}"
    )
    print(f"  Consumer B: отримав з redelivered={redelivered_flag['value']}")

await chaos_test_consumer_crash_triggers_redelivery()
print("✓ Chaos-тест — redelivery після crash: PASS")

In [ ]:
# Cleanup: зупиняємо контейнер після всіх тестів.
_rabbit.stop()
print("RabbitMQ container зупинено.")

## 5. Висновки

1. **Юніт-тести покривають всю гілкову логіку хендлера без брокера.** 4 тести × 4 гілки `PaymentHandler` ловлять регресію бізнес-логіки за мілісекунди, без Docker. Перевірка через `assert_awaited_once_with` / `assert_not_awaited` — Python-аналог `Received(1)` / `DidNotReceive()` з NSubstitute.

2. **TestContainers замінює моки на реальний RabbitMQ — єдиний рівень, що ловить bugs serialization, routing та ACK.** Юніт-тести з моками **ніколи** не побачать, що `Decimal` зник через `json.dumps`, або що routing key не збігається з binding. Pitfall — потрібен Docker; перший старт ~10–20с, тому групуємо тести через module-scope fixture (тут — один `_rabbit` на весь notebook).

3. **Chaos-тест на consumer crash — мінімальна страховка проти втрати повідомлень.** Реальні падіння в проді трапляються постійно (deploy, OOM, network blip). Якщо хендлер некоректно обробляє ACK — повідомлення або губляться, або множаться в DLQ. Перевіряємо `message.redelivered=True` після close-connection — це ~50 рядків коду, які рятують години розбору інцидентів.